In [250]:
import pandas as pd
from helpers import get_factor, get_price

In [251]:
prod = pd.read_csv("../production/yearly.csv")
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [252]:
plantlist2 = plantlist[["plantid", "energysource"]]

In [253]:
tmp0 = pd.merge(nat_mp, plantlist2, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [274]:
prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [275]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [276]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [277]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [278]:
tmp2 = pd.merge(tmp1, prod2, on="plantid")

In [285]:
coal_cost_per_t = 103.5 or 120
co2_cost = 70
electricity_price = 78.50
tmp2["revenue"] = tmp2["yearpower"] * electricity_price / 1000000

In [286]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [287]:
tmp2["profit"] = tmp2["revenue"] - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [288]:
co2s3.dtypes

plantid      object
amount_2    float64
dtype: object

In [289]:
tmp2.sort_values("profit")

,plantid,plantname,free_co2s,energysource,factor,fuel_price,amount_2,year,yearpower,revenue,co2_cost,coal_cost,profit
21,BB23020490,1MKA,0.0,Mineralölprodukte,2.30,75,3.015,2023,1182522,92.827977,211.05000,98.315217,-216.537240
17,BWpf-450-2948214-00000000,GKM Block 6,92027.0,Steinkohle,2.68,120,3.430,2023,2328376,182.777516,233.65811,153.582090,-204.462684
13,BWpf-450-2797933-00000000,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,Steinkohle,2.68,120,3.185,2023,2553799,200.473221,222.95000,142.611940,-165.088719
23,BB45025564,Kraftwerk Jänschwalde Block A,11960.0,Braunkohle,3.25,18,14.134,2023,11628593,912.844550,988.54280,78.280615,-153.978865
9,NW500-0342658,Scholven 1 DT,48161.0,Steinkohle,2.68,120,1.993,2023,1197970,94.040645,136.13873,89.238806,-131.336891
8,NW300-0877384,Weisweiler F,12223.0,Braunkohle,3.25,18,9.336,2023,7312086,573.998751,652.66439,51.707077,-130.372716
19,MV30000226,Kraftwerk Rostock Block A,14468.0,Steinkohle,2.68,120,1.728,2023,1145693,89.936900,119.94724,77.373134,-107.383474
0,NW300-0326774,Niederaußem G,26041.0,Braunkohle,3.25,18,13.311,2023,11725739,920.470512,929.94713,73.722462,-83.199080
3,BE166928,HKW Reuter West Dampfturbine D,63958.0,Steinkohle,2.68,120,1.547,2023,1284397,100.825164,103.81294,69.268657,-72.256432
5,NW500-0915123,Datteln 4,631.0,Steinkohle,2.68,120,2.945,2023,3406179,267.385051,206.10583,131.865672,-70.586450
